In [ ]:
# All package imports (run this cell first)
import sys
import subprocess
import json
from pathlib import Path

import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import PeftModel, get_peft_model, LoraConfig, TaskType

## Kernel check (use root .venv)

Run this cell first to confirm the notebook is using the project's root `.venv`.

In [ ]:
# Kernel verification
_venv_ok = "My-Crew-Manager" in sys.executable and ".venv" in sys.executable
print(f"Python: {sys.executable}")
print(f"Using root .venv: {'✓ Yes' if _venv_ok else '✗ No – select Kernel → Python (My-Crew-Manager .venv)'}")

# Model Part 1 – Proposal Summarization & Sprint Planning

From raw project description → structured Part 1 (summary, roles, features, goals, timeline).

Run from `AI/` or project root. Requires GOOGLE_API_KEY, OPENAI_API_KEY, or ANTHROPIC_API_KEY for Step 1.

## Setup paths

In [ ]:
# Resolve AI root
_cwd = Path.cwd()
_ai_root = _cwd if (_cwd / "llms").exists() else (_cwd / "AI" if (_cwd / "AI").exists() else _cwd)
if str(_ai_root) not in sys.path:
    sys.path.insert(0, str(_ai_root))

FINE_TUNE_DIR = _ai_root / "llms" / "fine_tune"
DATASET_DIR = FINE_TUNE_DIR / "dataset"
TOKENIZED_DIR = FINE_TUNE_DIR / "tokenized"
OUTPUT_DIR = FINE_TUNE_DIR / "qwen_model1_overview_lora"

print(f"AI root: {_ai_root}")
print(f"Dataset: {DATASET_DIR}")
print(f"Output: {OUTPUT_DIR}")

## Step 1: Generate dataset

Run build_synthetic to generate combined entries and split into model1 JSONL. Set GOOGLE_API_KEY, OPENAI_API_KEY, or ANTHROPIC_API_KEY.

In [ ]:
# Generate 20 entries by default (Gemini rate limit); use --count 500 with OpenAI/Anthropic
result = subprocess.run(
    [sys.executable, "-m", "llms.fine_tune.build_synthetic"],
    cwd=str(_ai_root),
    capture_output=False,
)
if result.returncode != 0:
    print("Build synthetic failed. Check API keys and try again.")
else:
    p = DATASET_DIR / "model1_description_to_part1.jsonl"
    count = len([ln for ln in p.read_text(encoding="utf-8").split("\n") if ln.strip()]) if p.exists() else 0
    print(f"model1_description_to_part1.jsonl: {count} examples")

## Step 2: Prepare tokenized dataset

In [ ]:
from llms.fine_tune.prepare_dataset import prepare_model1_dataset

tokenized_path = TOKENIZED_DIR / "tokenized_model1_qwen"
MAX_LENGTH = 512

if tokenized_path.exists():
    dataset = load_from_disk(str(tokenized_path))
    print(f"Loaded tokenized dataset from {tokenized_path}")
else:
    dataset = prepare_model1_dataset(
        model_name="qwen",
        max_length=MAX_LENGTH,
        output_dir=str(tokenized_path),
    )
print(f"Dataset size: {len(dataset)}")

## Step 3: Load model & apply LoRA

In [ ]:
MODEL_ID = "Qwen/Qwen2-0.5B-Instruct"
BATCH_SIZE = 2
EPOCHS = 3

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    trust_remote_code=True,
)
if torch.cuda.is_available():
    model = model.to("cuda")

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Step 4: Train

In [ ]:
import os
os.environ.setdefault("TENSORBOARD_LOGGING_DIR", str(OUTPUT_DIR / "logs"))

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()

## Step 5: Save adapter

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print(f"Saved adapter and tokenizer to {OUTPUT_DIR}")
print("Set PEFT_ADAPTER_PATH in AI/.env to use this adapter:")
print("  PEFT_ADAPTER_PATH=llms/fine_tune/qwen_model1_overview_lora")

## Step 6: Quick inference test

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    trust_remote_code=True,
)
model_infer = PeftModel.from_pretrained(base_model, str(OUTPUT_DIR))
if torch.cuda.is_available():
    model_infer = model_infer.to("cuda")

test_prompt = "Convert this project description to structured JSON with summary, roles, features, goals, timeline.\n\nDescription:\nBuild a task management web app for small teams with Kanban boards and real-time updates.\n\nOutput JSON only:"
inputs = tokenizer(test_prompt, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}
outputs = model_infer.generate(**inputs, max_new_tokens=384, do_sample=True, temperature=0.4)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Generated response:")
try:
    parsed = json.loads(response.strip())
    print(json.dumps(parsed, indent=2, ensure_ascii=False)[:800] + "...")
except json.JSONDecodeError:
    print(response[:800])